# 💼 01c — Pipeline catégories socio-professionnelles (CSP)

Construit `dim_csp.parquet`, une ligne par département. Lancer
`00_config_commun.ipynb` avant.

Source : INSEE, structure de la population active (15-64 ans) par catégorie
socioprofessionnelle, recensement 2022.
https://www.insee.fr/fr/statistiques/2012721#tableau-TCRD_014_tab1_departements
Fichier : `data/raw/insee_csp/insee_csp.xlsx`, feuille "DEP".

Table statique comme `dim_geo_pop` (pas de dimension temporelle), la structure
socioprofessionnelle évolue trop lentement pour justifier un suivi mensuel
sur 2020-2025.

7 catégories (nomenclature PCS INSEE), en part de la population active 15-64
ans, en décimales (`0.163` = 16,3 %) :

| dim_csp | Description des variables |
|---|---|
| `tx_agriculteurs` | Agriculteurs exploitants |
| `tx_artisans` | Artisans, commerçants, chefs d'entreprise |
| `tx_cadres` | Cadres, professions intellectuelles supérieures |
| `tx_prof_interm` | Professions intermédiaires |
| `tx_employes` | Employés |
| `tx_ouvriers` | Ouvriers |
| `tx_autres` | Autres (sans activité identifiée, etc.) |

La catégorie socioprofessionnelle est corrélée à des facteurs qui influencent
indirectement le recours aux urgences : exposition professionnelle
(agriculteurs/ouvriers → pollens, poussières, allergènes), couverture
complémentaire santé, éloignement des structures de soins, habitudes de
recours au système de santé.

In [1]:
# Préambule : on se place dans le répertoire racine du projet et on ajoute le répertoire courant au PYTHONPATH pour pouvoir importer src/config.py

# pour recharger automatiquement les modules modifiés （src config surtout） sans redémarrer le kernel
%load_ext autoreload 
%autoreload 2

import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS,  DEPT_NOM_TO_CODE
from src.validation import valider_dim_table

print(
    f"Config chargée depuis src/config.py : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"RAW_DIR    = {RAW_DIR}")
print(f"TABLES_DIR = {TABLES_DIR}")

Config chargée depuis src/config.py : 96 départements | 2020–2025
RAW_DIR    = /Users/siranh/Documents/Data Scientest/projet_liora/data/raw
TABLES_DIR = /Users/siranh/Documents/Data Scientest/projet_liora/data/processed


In [2]:
def build_dim_csp() -> pd.DataFrame:
    """
    Construit la table dim_csp à partir du fichier INSEE CSP (feuille "DEP").

    CLÉ PRIMAIRE : dept
    COLONNES : tx_agriculteurs, tx_artisans, tx_cadres, tx_prof_interm,
               tx_employes, tx_ouvriers, tx_autres  (en décimales, ex: 0.12)
    """
    fpath = RAW_DIR / "insee_csp" / "insee_csp.xlsx"
    if not fpath.exists():
        print(f"⚠️  Fichier manquant : {fpath}")
        return pd.DataFrame()

    df = pd.read_excel(fpath, sheet_name="DEP", header=None, skiprows=4)
    df.columns = ["dept", "nom", "tx_agriculteurs", "tx_artisans",
                  "tx_cadres", "tx_prof_interm", "tx_employes",
                  "tx_ouvriers", "tx_autres"]

    df["dept"] = df["dept"].astype(str).str.strip().str.zfill(2).str.upper()
    df = df[df["dept"].isin(DEPTS)]

    for col in df.columns[2:]:
        df[col] = pd.to_numeric(df[col], errors="coerce") / 100

    cols_out = ["dept"] + list(df.columns[2:])
    df = df[cols_out].reset_index(drop=True)

    print(f"✅ dim_csp : {len(df)} départements × {df.shape[1]} colonnes")
    return df


dim_csp = build_dim_csp()

if not dim_csp.empty:
    valider_dim_table(dim_csp, "dim_csp", cle=["dept"])
    dim_csp.to_parquet(TABLES_DIR / "dim_csp.parquet", index=False)
    print(f"\n Sauvegardé → data/processed/dim_csp.parquet")
    display(dim_csp.head(5))

✅ dim_csp : 96 départements × 8 colonnes
── Validation de dim_csp ──
  ✅ Tous les codes dept sont valides (96 départements)
  ✅ Aucun doublon sur la clé ['dept']
  ✅ OK — prêt pour la fusion (dim_csp)


 Sauvegardé → data/processed/dim_csp.parquet


,dept,tx_agriculteurs,tx_artisans,tx_cadres,tx_prof_interm,tx_employes,tx_ouvriers,tx_autres
0,01,0.009,0.065,0.163,0.271,0.253,0.232,0.007
1,02,0.017,0.049,0.087,0.228,0.293,0.303,0.025
2,03,0.030,0.065,0.096,0.229,0.305,0.262,0.013
3,04,0.027,0.101,0.124,0.258,0.278,0.203,0.010
4,05,0.027,0.096,0.107,0.282,0.299,0.183,0.006
